# Patrón de Comportamiento: Command

## Introducción
El patrón Command encapsula una petición como un objeto, permitiendo parametrizar clientes con diferentes peticiones, encolar o registrar peticiones y soportar operaciones deshacer.

## Objetivos
- Comprender cómo encapsular acciones como objetos.
- Identificar cuándo es útil el patrón Command.
- Comparar la solución con y sin el patrón.

## Ejemplo de la vida real
**Contexto: Control remoto de dispositivos inteligentes**
Un control remoto puede ejecutar diferentes comandos (encender luz, abrir puerta, etc.) y registrar un historial para deshacer acciones.

**¿Dónde se usa en proyectos reales?**
En sistemas de menús, controles remotos, editores con deshacer/rehacer, etc.

## Sin patrón Command (forma errónea)
El cliente debe conocer y ejecutar cada acción directamente.

In [1]:
class Luz:
    def encender(self) -> None:
        print('Luz encendida')
    def apagar(self) -> None:
        print('Luz apagada')

luz = Luz()
luz.encender()
luz.apagar()

Luz encendida
Luz apagada


## Con patrón Command (forma correcta)
Las acciones se encapsulan como comandos y el cliente puede ejecutarlas o deshacerlas.

In [2]:
import abc

class Command(abc.ABC):
    @abc.abstractmethod
    def ejecutar(self) -> None:
        ...
    @abc.abstractmethod
    def deshacer(self) -> None:
        ...

class EncenderLuz(Command):
    def __init__(self, luz: Luz) -> None:
        self.luz = luz
    def ejecutar(self) -> None:
        self.luz.encender()
    def deshacer(self) -> None:
        self.luz.apagar()

class ControlRemoto:
    def __init__(self) -> None:
        self.historial: list[Command] = []
    def ejecutar_comando(self, comando: Command) -> None:
        comando.ejecutar()
        self.historial.append(comando)
    def deshacer_ultimo(self) -> None:
        if self.historial:
            self.historial.pop().deshacer()

luz = Luz()
control = ControlRemoto()
cmd = EncenderLuz(luz)
control.ejecutar_comando(cmd)
control.deshacer_ultimo()

Luz encendida
Luz apagada


## UML del patrón Command
```plantuml
@startuml
interface Command {
    + ejecutar()
    + deshacer()
}
class EncenderLuz {
    + ejecutar()
    + deshacer()
}
class Luz {
    + encender()
    + apagar()
}
class ControlRemoto {
    + ejecutar_comando(comando)
    + deshacer_ultimo()
}
EncenderLuz --> Luz
ControlRemoto --> Command
@enduml
```

## Otro ejemplo de la vida real: Cola de trabajos en segundo plano (background jobs)
**Contexto:** sistemas como Celery o Sidekiq permiten encolar tareas (enviar un email, generar un reporte, procesar una imagen) para que se ejecuten de forma asíncrona, reintentando las que fallen sin que el código que las encoló tenga que ejecutar nada directamente.

### Sin patrón (forma errónea)
El cliente ejecuta cada tarea directamente e inmediatamente — no hay forma de encolarla, reintentarla o ejecutarla más tarde.

In [3]:
class ServicioEmail:
    def enviar(self, destinatario: str, asunto: str) -> None:
        print(f'Enviando email a {destinatario}: {asunto}')

class ServicioReportes:
    def generar(self, tipo: str) -> None:
        print(f'Generando reporte de {tipo}')

# El cliente ejecuta cada tarea de inmediato, sin poder encolarla ni reintentarla si falla
email = ServicioEmail()
email.enviar('ana@mail.com', 'Bienvenida')
reportes = ServicioReportes()
reportes.generar('ventas')

Enviando email a ana@mail.com: Bienvenida
Generando reporte de ventas


### Con patrón (forma correcta)
Cada tarea se encapsula como un `Job` (comando) con su propio `ejecutar()`. La `ColaTrabajos` puede encolarlos, procesarlos en orden y registrar cuáles fallaron para reintentarlos después.

In [4]:
import abc

class Job(abc.ABC):
    @abc.abstractmethod
    def ejecutar(self) -> None:
        ...

class EnviarEmailJob(Job):
    def __init__(self, servicio: ServicioEmail, destinatario: str, asunto: str) -> None:
        self.servicio = servicio
        self.destinatario = destinatario
        self.asunto = asunto
    def ejecutar(self) -> None:
        self.servicio.enviar(self.destinatario, self.asunto)

class GenerarReporteJob(Job):
    def __init__(self, servicio: ServicioReportes, tipo: str) -> None:
        self.servicio = servicio
        self.tipo = tipo
    def ejecutar(self) -> None:
        self.servicio.generar(self.tipo)


class ColaTrabajos:
    def __init__(self) -> None:
        self.pendientes: list[Job] = []
        self.fallidos: list[Job] = []
    def encolar(self, job: Job) -> None:
        self.pendientes.append(job)
    def procesar(self) -> None:
        while self.pendientes:
            job = self.pendientes.pop(0)
            try:
                job.ejecutar()
            except Exception:
                self.fallidos.append(job)


cola = ColaTrabajos()
cola.encolar(EnviarEmailJob(ServicioEmail(), 'ana@mail.com', 'Bienvenida'))
cola.encolar(GenerarReporteJob(ServicioReportes(), 'ventas'))
cola.procesar()

Enviando email a ana@mail.com: Bienvenida
Generando reporte de ventas


### UML del ejemplo de cola de trabajos
```plantuml
@startuml
abstract class Job {
    + ejecutar()
}
class EnviarEmailJob
class GenerarReporteJob
Job <|-- EnviarEmailJob
Job <|-- GenerarReporteJob
class ColaTrabajos {
    - pendientes: list
    - fallidos: list
    + encolar(job)
    + procesar()
}
ColaTrabajos --> Job
@enduml
```

### ¿Dónde más se usa Command?
- **Colas de trabajos en segundo plano:** exactamente este ejemplo — Celery, Sidekiq, AWS SQS encapsulan cada tarea como un objeto que se encola y ejecuta después.
- **Controles remotos y automatización del hogar:** el ejemplo con el que abre este notebook — encender luces, abrir puertas, cada uno como comando independiente.
- **Editores con deshacer/rehacer:** cada acción del usuario (mover, escribir, borrar) es un comando que sabe cómo revertirse.
- **Macros y grabación de acciones:** grabar una secuencia de comandos ejecutados para poder reproducirla exactamente igual más tarde.
- **Transacciones y sagas en microservicios:** cada paso de una transacción distribuida se modela como un comando con su propia lógica de compensación (deshacer) si un paso posterior falla.

**Ejercicio de reflexión:** ¿cómo implementarías un reintento automático (máximo 3 intentos) para los jobs en `self.fallidos`, sin modificar la clase `Job` ni sus subclases?

## Actividad
Crea comandos para abrir/cerrar una puerta y agrégalos al control remoto. Implementa la función de deshacer.

---
## Explicación de conceptos clave
- **Encapsulamiento de acciones:** Permite parametrizar y registrar acciones.
- **Deshacer/rehacer:** Facilita la implementación de historial de acciones.
- **Aplicación en la vida real:** Útil en controles remotos, menús y editores.

## Conclusión
El patrón Command es ideal para sistemas que requieren registrar, deshacer o parametrizar acciones.